# Co-Smoothing Evaluation: Held-Out Bits per Spike

This notebook is the **co-smoothing** evaluation for GPFA latent variable
models, part of the broader latent-dynamics evaluation suite (see the repo
README for how this fits alongside leave-neuron-out MSE and behavioral
decoding).

**Question this notebook answers:** does the fitted GPFA model actually
explain the *raw spiking data* well, on neurons and trials it never saw during
fitting — or could a low-dimensional model with strong prediction error scores
still be missing (or hallucinating) structure in the spikes themselves?

**Metric:** *co-smoothing*, scored in **bits per spike (bps)**, following the
convention from the [Neural Latents Benchmark '21](https://arxiv.org/abs/2109.04463)
(NLB '21). Bits/spike measures the log-likelihood gain of a model's spike-count
predictions over a constant-rate ("null") baseline, normalized by the number of
spikes and converted to bits (dividing by `log(2)`). A score of 0 means the
model is no better than predicting the neuron's mean firing rate; positive
scores mean the model's predictions capture real structure in when/how much
that neuron fires.

**Approach — the "bridge" method:**

1. Split neurons into **held-in** (used to fit GPFA) and **held-out** (only used
   for evaluation), and split trials into **train** and **test**.
2. Fit GPFA on held-in neurons, training trials only, and extract orthonormalized
   latent trajectories.
3. Fit a linear regression bridging GPFA latents → held-in neurons' observed
   spike counts (this converts latents into a rate-like signal in "spike count"
   units).
4. Fit one Poisson GLM per held-out neuron, bridging the held-in rate
   predictions from step 3 → that held-out neuron's actual spike counts.
5. Score each held-out neuron's GLM predictions on the test trials using bits/spike.

This gives a co-smoothing score for held-out neurons the GPFA model never saw,
which is what makes it a meaningful check on whether the latents generalize to
new neurons — not just new trials.

**Sections**

1. Setup and data loading
2. Held-in/held-out neuron split and train/test trial split
3. Diagnostics: firing-rate and spike-count distributions
4. Co-smoothing bridge method: helper functions
5. Cross-validate latent dimensionality by co-smoothing score
6. Robustness check: sensitivity to low-firing held-in neurons

## 1. Setup

Import dependencies and add the project root to the path so the local
`Analysis_library` package and `data_paths` module can be imported.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import neo
import quantities as pq
from elephant.gpfa import GPFA

notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data_paths import DATA_PATH
from Analysis_library.file_loading import SessionData
import Analysis_library.analysis as ana_lib

## 2. Load Session Data, Select Units, Build GPFA Trials

Same data-preparation steps as the other notebooks in this repo: load the
session, select well-isolated units above a minimum spike-count threshold, and
build per-trial `neo.SpikeTrain` objects aligned to trial onset. See the main
`GPFA_data.ipynb` notebook for a more detailed breakdown of these steps.

In [ ]:
SESSION_KEY = "332"
sessions = [SessionData(DATA_PATH[SESSION_KEY])]
s = sessions[0]

print("Session:", s.mat_file)
print("Number of units:", len(s.units_dict["spikes"]))
print("Number of trials:", len(s.trials_dict["t"]))

In [ ]:
SAMPLE_RATE_HZ = 30000.0
BIN_SIZE_MS = 30 # Consider using a smaller bin size (e.g., 10 ms) for better temporal resolution, but this may increase computational load.
PRE_TRIAL_S = 1.0
POST_TRIAL_S = 4.0

good_units = ana_lib.get_good_units(s.units_dict, verbose=False)
good_units = np.asarray(good_units, dtype=int)
print("Number of good units:", len(good_units))
trial_times = np.asarray(s.trials_dict["t"], dtype=float)

print("\nFirst 3 trial times:")
print(trial_times[:3])
# shape: trial x neuron x time bins

spikes = s.units_dict["spikes"]
TRIAL_DURATION_S = PRE_TRIAL_S + POST_TRIAL_S

# Count spikes for each unit across all trial windows
spike_counts_per_unit = []

for unit_idx in good_units:
    total_spikes = 0

    for trial_time in trial_times:
        trial_start = trial_time - PRE_TRIAL_S
        trial_stop = trial_time + POST_TRIAL_S
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        total_spikes += len(trial_spikes)
    spike_counts_per_unit.append(total_spikes)
spike_counts_per_unit = np.asarray(spike_counts_per_unit)

# Keep units with at least 5 spikes across trial windows
MIN_SPIKES = 5
gpfa_units = good_units[spike_counts_per_unit >= MIN_SPIKES]

print("Original good units:", len(good_units))
print("Units kept for GPFA:", len(gpfa_units))

In [ ]:
# Build GPFA trials

gpfa_data = []

for trial_time in trial_times:
    trial_spiketrains = []
    trial_start = trial_time - PRE_TRIAL_S
    trial_stop = trial_time + POST_TRIAL_S

    for unit_idx in gpfa_units:
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        trial_spikes = (trial_spikes - trial_start)
        spike_train = neo.SpikeTrain(
            trial_spikes * pq.s,
            t_start=0 * pq.s,
            t_stop=TRIAL_DURATION_S * pq.s
        )
        trial_spiketrains.append(spike_train)
    
    gpfa_data.append(trial_spiketrains)

print("Number of neurons per trial:", len(gpfa_data[0]))
print(s.data.keys())

## 3. Bin Spike Trains and Set Candidate Dimensionalities

Set the candidate latent dimensionalities to evaluate (`X_DIMS`) and convert
each trial's spike trains into binned spike counts
(`trials x neurons x time bins`) at `BIN_SIZE_MS` resolution.

In [ ]:
from sklearn.model_selection import KFold
from elephant.conversion import BinnedSpikeTrain
import numpy as np
import time

X_DIMS = [1, 2, 3, 4, 5, 8, 10, 15, 20] # [2, 4, 6, 8]
N_FOLDS = 4
RANDOM_SEED = 42

binned_trials = []

# trials x neurons x time bins
for trial in gpfa_data:
    binned = BinnedSpikeTrain(trial, bin_size=BIN_SIZE_MS * pq.ms)
    counts = binned.to_array()
    binned_trials.append(counts)
binned_trials = np.asarray(binned_trials)

print( "Binned data shape:", binned_trials.shape)

## 4. Held-In / Held-Out Neuron Split and Train/Test Trial Split

Co-smoothing requires two independent splits:

- **Neuron split** — `heldin_idx` (used to fit GPFA and the latent→rate bridge)
  vs. `heldout_idx` (only ever scored, never used to fit anything)
- **Trial split** — `train_trial_idx` (used for fitting and cross-validation)
  vs. `test_trial_idx` (a completely untouched set, scored only once at the end)

Both are random splits fixed by `RANDOM_SEED` for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

# Random neuron split
N_NEURONS = binned_trials.shape[1]

heldin_idx, heldout_idx = train_test_split(np.arange(N_NEURONS), test_size=0.30, random_state=RANDOM_SEED, shuffle=True)
heldin_idx = np.sort(heldin_idx)
heldout_idx = np.sort(heldout_idx)
print(f"Held-in neurons : {len(heldin_idx)}")
print(f"Held-out neurons: {len(heldout_idx)}")

# Random trial split
train_trial_idx, test_trial_idx = train_test_split(np.arange(len(gpfa_data)), test_size=0.20, random_state=RANDOM_SEED, shuffle=True)
print(f"Training trials : {len(train_trial_idx)}")
print(f"Testing trials  : {len(test_trial_idx)}")

## 5. Diagnostics: Firing-Rate and Spike-Count Distributions

Before running the co-smoothing pipeline, it's worth checking whether the
held-in and held-out neuron populations look similar, and where low-firing
neurons fall relative to the rest of the population — low-firing neurons are
noisy to score (few spikes means a noisy bits/spike estimate) and are the
target of the robustness check in Section 7.

### 5.1 Held-in vs. held-out firing rates

Log-scale and linear-zoomed histograms of firing rate (Hz) for held-in vs.
held-out neurons, with dashed lines marking low-firing percentile cutoffs
(0/1/5/10/15/20%) computed on the held-out population.

In [ ]:
# %%
# Firing-rate distributions: held-in vs held-out, with filtering quantile markers

TRIAL_DURATION_S = PRE_TRIAL_S + POST_TRIAL_S
print(f"Trial window duration (fixed by construction): {TRIAL_DURATION_S:.2f} s")

inter_trial_intervals = np.diff(trial_times)
print(f"Average time between trial onsets: {inter_trial_intervals.mean():.2f} s "
      f"(min {inter_trial_intervals.min():.2f}, max {inter_trial_intervals.max():.2f})")

n_train_trials = len(train_trial_idx)

heldin_spike_totals = binned_trials[np.ix_(train_trial_idx, heldin_idx)].sum(axis=(0, 2))
heldout_spike_totals = binned_trials[np.ix_(train_trial_idx, heldout_idx)].sum(axis=(0, 2))

heldin_rate_hz = heldin_spike_totals / (n_train_trials * TRIAL_DURATION_S)
heldout_rate_hz = heldout_spike_totals / (n_train_trials * TRIAL_DURATION_S)

print(f"Held-in  neurons: {len(heldin_idx)}, mean = {heldin_rate_hz.mean():.2f} Hz, median = {np.median(heldin_rate_hz):.2f} Hz")
print(f"Held-out neurons: {len(heldout_idx)}, mean = {heldout_rate_hz.mean():.2f} Hz, median = {np.median(heldout_rate_hz):.2f} Hz")

percentiles = [0, 1, 5, 10, 15, 20]
quantile_rates = {p: np.percentile(heldout_rate_hz, p) for p in percentiles}

# --- log-scale view (best for seeing where the low-firing quantiles actually fall) ---
'''
Log scale: compresses the x-axis so a neuron at 0.1 Hz and a neuron at 15 Hz can both be seen clearly on one plot without one end being crushed into nothing. 
Good for seeing the entire range and where percentile cutoffs sit relative to the full population, outliers included.
'''
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
eps = 1e-3  # floor so log(0) doesn't blow up; a handful of neurons may have 0 Hz on train trials
log_bins = np.logspace(np.log10(eps), np.log10(max(heldin_rate_hz.max(), heldout_rate_hz.max())), 50)
ax.hist(np.clip(heldin_rate_hz, eps, None), bins=log_bins, alpha=0.5, label=f"Held-in (n={len(heldin_idx)})", color="tab:blue")
ax.hist(np.clip(heldout_rate_hz, eps, None), bins=log_bins, alpha=0.5, label=f"Held-out (n={len(heldout_idx)})", color="tab:orange")
ax.set_xscale("log")

ymax = ax.get_ylim()[1]
# stagger label heights so close-together quantile lines stay legible
for i, (p, val) in enumerate(sorted(quantile_rates.items(), key=lambda kv: kv[1])):
    plot_val = max(val, eps)
    ax.axvline(plot_val, color="red", ls="--", lw=1)
    y = ymax * (0.95 - 0.08 * i)
    ax.text(plot_val, y, f"{p}%", color="red", rotation=90, va="top", ha="right", fontsize=8)

ax.set_xlabel("Firing rate (Hz, log scale)")
ax.set_ylabel("Number of neurons")
ax.set_title("Log-scale view")
ax.legend()
ax.grid(alpha=0.3, which="both")

# --- linear zoomed view, capped near the bulk of the distribution ---
'''
Linear zoom (0 to 95th percentile): clips off those high-firing outliers entirely and stretches out just the bulk (your 0-2 Hz mass) so you can see 
the actual shape and separation between held-in/held-out in the region that matters — since your first histogram showed almost everything crammed under 2 Hz anyway.
'''
ax = axes[1]
zoom_max = np.percentile(np.concatenate([heldin_rate_hz, heldout_rate_hz]), 95)
bins = np.linspace(0, zoom_max, 40)
ax.hist(heldin_rate_hz, bins=bins, alpha=0.5, label=f"Held-in (n={len(heldin_idx)})", color="tab:blue")
ax.hist(heldout_rate_hz, bins=bins, alpha=0.5, label=f"Held-out (n={len(heldout_idx)})", color="tab:orange")

ymax = ax.get_ylim()[1]
for i, (p, val) in enumerate(sorted(quantile_rates.items(), key=lambda kv: kv[1])):
    if val > zoom_max:
        continue
    ax.axvline(val, color="red", ls="--", lw=1)
    y = ymax * (0.95 - 0.08 * i)
    ax.text(val, y, f"{p}%", color="red", rotation=90, va="top", ha="right", fontsize=8)

ax.set_xlim(0, zoom_max)
ax.set_xlabel("Firing rate (Hz)")
ax.set_ylabel("Number of neurons")
ax.set_title(f"Linear zoom (0–{zoom_max:.1f} Hz, 95th pctile)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nHeld-out rate quantiles used for filtering:")
for p, val in quantile_rates.items():
    print(f"  {p}%: {val:.3f} Hz")

### 5.2 Held-in vs. held-out spike counts

Same comparison as above, but in raw total spike count (over training trials)
rather than firing rate — useful since bits/spike is directly sensitive to how
many spikes a neuron contributes.

In [ ]:
# %%
# Spike-count distributions: held-in vs held-out, with filtering quantile markers

heldin_counts = heldin_spike_totals  # already computed above, on train_trial_idx
heldout_counts = heldout_spike_totals

print(f"Held-in  neurons: {len(heldin_idx)}, mean = {heldin_counts.mean():.1f} spikes, median = {np.median(heldin_counts):.1f} spikes")
print(f"Held-out neurons: {len(heldout_idx)}, mean = {heldout_counts.mean():.1f} spikes, median = {np.median(heldout_counts):.1f} spikes")

percentiles = [0, 1, 5, 10, 15, 20]
quantile_counts = {p: np.percentile(heldout_counts, p) for p in percentiles}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# log-scale view
ax = axes[0]
eps = 0.5  # floor for log display; 0-spike neurons can't be shown on a log axis
log_bins = np.logspace(np.log10(eps), np.log10(max(heldin_counts.max(), heldout_counts.max())), 50)
ax.hist(np.clip(heldin_counts, eps, None), bins=log_bins, alpha=0.5, label=f"Held-in (n={len(heldin_idx)})", color="tab:blue")
ax.hist(np.clip(heldout_counts, eps, None), bins=log_bins, alpha=0.5, label=f"Held-out (n={len(heldout_idx)})", color="tab:orange")
ax.set_xscale("log")

ymax = ax.get_ylim()[1]
for i, (p, val) in enumerate(sorted(quantile_counts.items(), key=lambda kv: kv[1])):
    plot_val = max(val, eps)
    ax.axvline(plot_val, color="red", ls="--", lw=1)
    y = ymax * (0.95 - 0.06 * i)
    ax.text(plot_val, y, f"{p}%", color="red", rotation=90, va="top", ha="right", fontsize=8)

ax.set_xlabel("Total spike count (log scale)")
ax.set_ylabel("Number of neurons")
ax.set_title("Log-scale view")
ax.legend()
ax.grid(alpha=0.3, which="both")

# linear zoomed view
ax = axes[1]
zoom_max = np.percentile(np.concatenate([heldin_counts, heldout_counts]), 95)
bins = np.linspace(0, zoom_max, 40)
ax.hist(heldin_counts, bins=bins, alpha=0.5, label=f"Held-in (n={len(heldin_idx)})", color="tab:blue")
ax.hist(heldout_counts, bins=bins, alpha=0.5, label=f"Held-out (n={len(heldout_idx)})", color="tab:orange")

ymax = ax.get_ylim()[1]
for i, (p, val) in enumerate(sorted(quantile_counts.items(), key=lambda kv: kv[1])):
    if val > zoom_max:
        continue
    ax.axvline(val, color="red", ls="--", lw=1)
    y = ymax * (0.95 - 0.06 * i)
    ax.text(val, y, f"{p}%", color="red", rotation=90, va="top", ha="right", fontsize=8)

ax.set_xlim(0, zoom_max)
ax.set_xlabel("Total spike count")
ax.set_ylabel("Number of neurons")
ax.set_title(f"Linear zoom (0–{zoom_max:.0f} spikes, 95th pctile)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nHeld-out spike-count quantiles:")
for p, val in quantile_counts.items():
    print(f"  {p}%: {val:.1f} spikes")

### 5.3 Held-in firing-rate distribution alone

A closer look at just the held-in population's mean firing rate across
training trials, again with percentile markers — this is the distribution the
Section 7 robustness sweep filters against.

In [ ]:
# %%
# Held-in mean firing rate (training trials), log scale with percentile markers

n_train_trials = len(train_trial_idx)
per_trial_rates_heldin = (
    binned_trials[np.ix_(train_trial_idx, heldin_idx)].sum(axis=2) / TRIAL_DURATION_S
)
heldin_mean_rate_hz = per_trial_rates_heldin.mean(axis=0)

percentiles = [0, 1, 5, 10, 15, 20]
quantile_rates = {p: np.percentile(heldin_mean_rate_hz, p) for p in percentiles}

eps = 1e-3  # floor for log display; 0 Hz neurons get clipped to this
plt.figure(figsize=(7, 5))
log_bins = np.logspace(np.log10(eps), np.log10(heldin_mean_rate_hz.max()), 50)
plt.hist(np.clip(heldin_mean_rate_hz, eps, None), bins=log_bins, color="tab:blue", alpha=0.7)
plt.xscale("log")

ymax = plt.ylim()[1]
for i, (p, val) in enumerate(sorted(quantile_rates.items(), key=lambda kv: kv[1])):
    plot_val = max(val, eps)
    plt.axvline(plot_val, color="red", ls="--", lw=1)
    plt.text(plot_val, ymax * (0.95 - 0.06*i), f"{p}%", color="red", rotation=90, va="top", ha="right", fontsize=8)

plt.xlabel("Mean firing rate across training trials (Hz, log scale)")
plt.ylabel("Number of held-in neurons")
plt.title("Held-in firing rates (training trials, log scale)")
plt.grid(alpha=0.3, which="both")
plt.show()

for p, val in quantile_rates.items():
    print(f"{p}%: {val:.3f} Hz")

## 6. Co-Smoothing Bridge Method

### 6.1 Helper functions

- `bits_per_spike` — the core NLB co-bps metric: log-likelihood gain of the
  model's Poisson rate predictions over a constant-rate null model, normalized
  by total spike count and converted to bits.
- `subset_gpfa_data` — pulls a `(trials, neurons)` block of spike trains out of
  `gpfa_data` for a given trial/neuron index set.
- `flatten` — reshapes per-trial latent trajectories and binned spike counts
  into flat `(time bins, x_dim)` / `(time bins, neurons)` arrays for regression.

In [ ]:
from scipy.stats import poisson
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.model_selection import KFold

def bits_per_spike(rate_pred, spike_true, eps=1e-9):
    """Standard NLB co-bps: log-likelihood gain over a constant-rate null model."""
    rate_pred = np.clip(rate_pred, eps, None)
    ll_model = poisson.logpmf(spike_true, rate_pred).sum()
    null_rate = max(spike_true.mean(), eps)
    ll_null = poisson.logpmf(spike_true, null_rate).sum()
    n_spikes = spike_true.sum()
    return (ll_model - ll_null) / (n_spikes * np.log(2)) if n_spikes > 0 else np.nan

def subset_gpfa_data(trial_idx, neuron_idx):
    """Pull out a (trials x neurons) block of neo SpikeTrains from gpfa_data."""
    return [[gpfa_data[t][n] for n in neuron_idx] for t in trial_idx]

def flatten(factors_list, counts_subset):
    """
    factors_list: output of gpfa_model.transform(..., returned_data=['latent_variable_orth'])
                  either an object array of (x_dim, T) arrays, or a dict with that key
    counts_subset: (n_trials, n_neurons, T) binned counts
    """
    if isinstance(factors_list, dict):
        factors_list = factors_list['latent_variable_orth']
    X = np.concatenate([f.T for f in factors_list], axis=0)
    Y = np.concatenate([counts_subset[i].T for i in range(counts_subset.shape[0])], axis=0)
    return X, Y

### 6.2 The bridge pipeline

`run_gpfa_bridge` implements the full method described in the overview: fit
GPFA on held-in neurons and training trials, bridge latents → held-in rates via
linear regression, bridge held-in rates → each held-out neuron's spikes via a
Poisson GLM, and score each held-out neuron's test-trial predictions in
bits/spike. Returns both the mean score across held-out neurons (the headline
number for a given `x_dim` / train-test split) and the raw per-neuron scores
for diagnostics.

In [ ]:
def run_gpfa_bridge(x_dim, train_idx, test_idx):
    # Step A: fit GPFA on held-in neurons, training trials only (TRIAL SPLIT)
    st_train = subset_gpfa_data(train_idx, heldin_idx)
    st_test  = subset_gpfa_data(test_idx,  heldin_idx)

    gpfa_model = GPFA(bin_size=BIN_SIZE_MS * pq.ms, x_dim=x_dim)
    gpfa_model.fit(st_train)
    
    fac_train = gpfa_model.transform(st_train, returned_data=['latent_variable_orth'])
    fac_test  = gpfa_model.transform(st_test,  returned_data=['latent_variable_orth'])

    # Step B: linear regression, factors -> held-in observed rates (raw counts)
    X_train, Y_heldin_train = flatten(fac_train, binned_trials[np.ix_(train_idx, heldin_idx)])
    X_test,  Y_heldin_test  = flatten(fac_test,  binned_trials[np.ix_(test_idx,  heldin_idx)])

    lin_reg = LinearRegression().fit(X_train, Y_heldin_train)
    rate_pred_train = np.clip(lin_reg.predict(X_train), 1e-3, None)  # rectify non-positive
    rate_pred_test  = np.clip(lin_reg.predict(X_test),  1e-3, None)

    # Step C: Poisson GLM, held-in rate predictions -> held-out spikes (one GLM per held-out neuron)
    _, Y_heldout_train = flatten(fac_train, binned_trials[np.ix_(train_idx, heldout_idx)])
    _, Y_heldout_test  = flatten(fac_test,  binned_trials[np.ix_(test_idx,  heldout_idx)])

    bps_per_neuron = []
    for n in range(len(heldout_idx)):
        glm = PoissonRegressor(alpha=1e-3, max_iter=500)
        glm.fit(rate_pred_train, Y_heldout_train[:, n])
        lam_test = glm.predict(rate_pred_test)
        bps_per_neuron.append(bits_per_spike(lam_test, Y_heldout_test[:, n]))

    return np.nanmean(bps_per_neuron), bps_per_neuron 
    # Average across held-out neurons: this is our one headline number for this x_dim/train-test split, 
    # plus the raw per-neuron list for diagnostics

### 6.3 Filter low-spike held-out neurons

Held-out neurons with very few spikes on the test trials give unreliable
bits/spike estimates (the denominator in `bits_per_spike` is the total spike
count), so neurons below `MIN_TEST_SPIKES` are dropped from evaluation before
running cross-validation.

In [ ]:
# Outlier neuron filter in evaluation set

heldout_spike_totals_test = binned_trials[np.ix_(test_trial_idx, heldout_idx)].sum(axis=(0, 2))
MIN_TEST_SPIKES = 5
keep_mask = heldout_spike_totals_test >= MIN_TEST_SPIKES

print(f"Dropping {(~keep_mask).sum()} of {len(heldout_idx)} held-out neurons for low spike counts")
heldout_idx = heldout_idx[keep_mask]
print("New held-out count:", len(heldout_idx))

## 7. Cross-Validate Latent Dimensionality by Co-Smoothing Score

For each candidate `x_dim`, run the bridge pipeline across `N_FOLDS`
cross-validation folds over the training trials, and average bits/spike across
folds. The best `x_dim` is the one with the highest mean CV bits/spike. That
dimensionality is then evaluated once on the fully held-out test trials for an
unbiased final score.

**Note:** this cell can be slow — for every fold and every `x_dim` it fits a
full GPFA model, a linear regression, and one Poisson GLM per held-out neuron.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_results = {}

for x_dim in X_DIMS:
    fold_bps = []
    for tr_i, val_i in kf.split(train_trial_idx):
        tr_trials  = train_trial_idx[tr_i]
        val_trials = train_trial_idx[val_i]
        mean_bps, _ = run_gpfa_bridge(x_dim, tr_trials, val_trials)
        fold_bps.append(mean_bps)
    cv_results[x_dim] = np.nanmean(fold_bps)
    print(f"x_dim={x_dim:>2d}  mean CV bits/spike = {cv_results[x_dim]:.4f}")

best_x_dim = max(cv_results, key=cv_results.get)
print("\nBest x_dim:", best_x_dim)

final_mean_bps, final_bps_per_neuron = run_gpfa_bridge(best_x_dim, train_trial_idx, test_trial_idx)
print("Held-out test bits/spike (mean):", final_mean_bps)
print("Held-out test bits/spike (median):", np.median(final_bps_per_neuron))

### 7.1 Visualize CV curve and final per-neuron scores

Left panel: mean CV bits/spike vs. candidate dimensionality, with the selected
`best_x_dim` marked. Right panel: the final held-out test set's per-neuron
bits/spike scores, sorted, for `best_x_dim`.

> **Known issue:** the per-`x_dim` printout below the plots
> (`for d in X_DIMS: np.mean(final_bps_per_neuron[d])`) doesn't do what it
> looks like — `final_bps_per_neuron` is the flat list of per-neuron scores
> from the *single* final run at `best_x_dim`, not a dict keyed by `x_dim`, so
> `final_bps_per_neuron[d]` just indexes into that list positionally rather
> than looking up scores for dimensionality `d`. The CV curve above (`xs`/`ys`
> from `cv_results`) is the correct per-dimensionality comparison; this
> printout is left as-is from the original notebook and should be treated as
> informational only, or removed/fixed before relying on its numbers.

In [ ]:
# Regenerated plots 

per_neuron = np.asarray(final_bps_per_neuron)
order = np.argsort(per_neuron)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
xs = sorted(cv_results.keys())
ys = [cv_results[x] for x in xs]
ax.plot(xs, ys, marker='o')
ax.axhline(0, color='k', lw=0.8)
ax.axvline(best_x_dim, color='r', ls='--', lw=1, label=f'best x_dim={best_x_dim}')
ax.set_xlabel("x_dim"); ax.set_ylabel("mean CV bits/spike"); ax.set_title("CV curve")
ax.legend()

ax = axes[1]
ax.bar(range(len(order)), per_neuron[order])
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel("held-out neuron (sorted)"); ax.set_ylabel("bits/spike")
ax.set_title(f"Final test set, x_dim={best_x_dim}, mean={final_mean_bps:.4f}")

plt.tight_layout()
plt.show()

print("Final held-out test bits/spike by latent dimensionality\n")

for d in X_DIMS:
    mean_bps = np.mean(final_bps_per_neuron[d])
    print(
        f"x_dim = {d:>2}: "
        f"mean bits/spike = {mean_bps:.4f}, "
    )

## 8. Robustness Check: Sensitivity to Low-Firing Held-In Neurons

Held-in neurons with very low firing rates contribute little signal to the
GPFA fit but add noise. This section checks how sensitive the final co-smoothing
score is to progressively removing the lowest-firing held-in neurons, at a
fixed dimensionality (`FIXED_X_DIM`).

### 8.1 Recompute held-in firing rates

Recomputed here (rather than reused from Section 5.3) since `heldin_idx` may
have changed since then — this cell is the source of truth for the rate array
the filtering sweep below uses.

In [ ]:
# Recompute held-in firing rates for current held-in neurons

heldin_counts = binned_trials[
    np.ix_(train_trial_idx, heldin_idx)
]

# total spikes / total seconds
heldin_mean_rate_hz = (
    heldin_counts.sum(axis=(0,2))
    /
    (len(train_trial_idx) * TRIAL_DURATION_S)
)

print("heldin_idx:", len(heldin_idx))
print("rate array:", len(heldin_mean_rate_hz))

### 8.2 Sweep firing-rate cutoffs

For each percentile cutoff, drop held-in neurons below that firing-rate
percentile, rerun the bridge pipeline at `FIXED_X_DIM`, and record the
resulting mean bits/spike. `original_heldin_idx` is restored at the end so
later cells aren't left with a filtered `heldin_idx`.

In [ ]:
FIXED_X_DIM = 10

percentiles = [0, 1, 5, 10, 15, 20]
percentile_results = {}
original_heldin_idx = heldin_idx.copy()

for p in percentiles:
    if p == 0:
        filtered_idx = original_heldin_idx
    else:
        threshold = np.percentile(heldin_mean_rate_hz, p)
        keep_mask = heldin_mean_rate_hz >= threshold
        filtered_idx = original_heldin_idx[keep_mask]
    heldin_idx = filtered_idx
    mean_bps, _ = run_gpfa_bridge(FIXED_X_DIM, train_trial_idx, test_trial_idx)
    percentile_results[p] = mean_bps
    print(f"Removed lowest {p}% held-in ({len(filtered_idx)} kept): {mean_bps:.4f} bits/spike")

heldin_idx = original_heldin_idx  # restore

### 8.3 Plot robustness curve

Mean held-out bits/spike as a function of how many low-firing held-in neurons
were removed — a flat curve suggests the co-smoothing score is robust to
excluding weak held-in neurons; a rising or falling trend suggests the model
is sensitive to them.

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(
    list(percentile_results.keys()),
    list(percentile_results.values()),
    marker="o"
)
plt.xlabel("Lowest firing neurons removed (%)")
plt.ylabel("Mean held-out bits/spike")
plt.title("Robustness of co-smoothing to low-firing neurons")
plt.grid(alpha=0.3)
plt.show()

### 8.4 Final evaluation with chosen cutoff

Apply the cutoff chosen from the robustness sweep (`CHOSEN_CUTOFF_PERCENT`) and
report the final held-out bits/spike at `FIXED_X_DIM` with that filtered
held-in neuron set.

In [ ]:
# %%
CHOSEN_CUTOFF_PERCENT = 2.5

threshold = np.percentile(heldin_mean_rate_hz, CHOSEN_CUTOFF_PERCENT)
keep_mask = heldin_mean_rate_hz >= threshold
heldin_idx = original_heldin_idx[keep_mask]

print(f"Kept {len(heldin_idx)} of {len(original_heldin_idx)} held-in neurons (removed bottom {CHOSEN_CUTOFF_PERCENT}%)")

final_mean_bps, final_bps_per_neuron = run_gpfa_bridge(FIXED_X_DIM, train_trial_idx, test_trial_idx)
print(f"Final held-out bits/spike (x_dim={FIXED_X_DIM}, filtered held-in): {final_mean_bps:.4f}")